In [1]:
import pandas as pd
import datetime as dt
from dataretrieval import nwis
from dataretrieval import waterdata
from Python_Files import data
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [2]:
upstream_id = "10155200"
downstream_id = "10155500"
start_date = '2016-01-01'
end_date = '2025-12-31'

discharge = data.get_discharge(upstream_id,downstream_id,start_date,end_date)
width = data.get_width()

In [56]:
possible_validation = pd.merge(discharge,cleaning,how='inner',on='Date').dropna()
# sample_daily["Downstream Width (ft)"] = sample_daily["Downstream Width (ft)"].ffill().bfill()
possible_validation.head(30)

,Upstream Mean Discharge (cfs),Downstream Mean Discharge (cfs),Upstream Width (ft),Downstream Width (ft),Upstream Depth (ft),Downstream Depth (ft)
Date,,,,,,
2016-02-26,137.0,168.0,65.0,58.0,1.325,4.180
2016-05-12,258.0,262.0,61.0,58.5,1.550,4.600
2016-06-24,846.0,996.0,64.0,62.1,2.735,7.135
2016-06-27,148.0,173.0,64.5,59.0,1.355,4.245
2016-09-06,572.0,619.0,61.0,61.4,2.070,5.645
2016-10-18,130.0,165.0,64.0,59.5,1.290,4.205
2017-03-15,152.0,210.0,55.0,53.0,1.845,4.835
2017-07-20,356.0,386.0,61.0,56.7,1.745,4.995
2017-10-11,151.0,167.0,63.4,57.2,1.295,4.145


In [3]:
datum, up_MSE, down_MSE = data.to_datum(discharge)
print(f"Upstream Training MSE: {up_MSE:.4f}")
print(f"DownstreamTraining MSE: {down_MSE:.4f}")

datum.tail(10)



Upstream Training MSE: 0.0018
DownstreamTraining MSE: 0.0007


,Upstream Mean Discharge (cfs),Upstream Datum (ft),Downstream Mean Discharge (cfs),Downstream Datum (ft)
Date,,,,
2025-12-18,135.0,0.808600,166.0,3.527459
2025-12-23,136.0,0.811514,163.0,3.509544
2025-12-24,136.0,0.811514,163.0,3.509544
2025-12-25,136.0,0.811514,168.0,3.539274
2025-12-26,136.0,0.811514,169.0,3.545143
2025-12-27,139.0,0.820188,181.0,3.613696
2025-12-28,137.0,0.814416,173.0,3.568373
2025-12-29,134.0,0.805676,159.0,3.485282
2025-12-30,140.0,0.823059,163.0,3.509544


In [4]:
years = datum.index.year.unique()
print(years)

Index([2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], dtype='int32', name='Date')


In [13]:
#continuous gage depth below surface? every2years
#Can't be in data.py due to weak processing power, bear with the multiple cells 

#2016-2017
up_2016_2017, m = waterdata.get_continuous(monitoring_location_id= f"USGS-{upstream_id}",
                                        parameter_code='00065',
                                        time=f"{years[0]}-01-01/{years[1]}-12-31",
                                        properties=["time","value","unit_of_measure"])
up_2016_2017.set_index("time",inplace=True)
up_2016_2017.index = up_2016_2017.index.date
up_2016_2017.index = pd.to_datetime(up_2016_2017.index)
up_2016_2017 = up_2016_2017.resample('D').first()

down_2016_2017, m = waterdata.get_continuous(monitoring_location_id= f"USGS-{downstream_id}",
                                        parameter_code='00065',
                                        time=f"{years[0]}-01-01/{years[1]}-12-31",
                                        properties=["time","value","unit_of_measure"])
down_2016_2017.set_index("time",inplace=True)
down_2016_2017.index = down_2016_2017.index.date
down_2016_2017.index = pd.to_datetime(down_2016_2017.index)
down_2016_2017 = down_2016_2017.resample('D').first()

In [14]:
#2018-2019
up_2018_2019, m = waterdata.get_continuous(monitoring_location_id= f"USGS-{upstream_id}",
                                        parameter_code='00065',
                                        time=f"{years[2]}-01-01/{years[3]}-12-31",
                                        properties=["time","value","unit_of_measure"])
up_2018_2019.set_index("time",inplace=True)
up_2018_2019.index = up_2018_2019.index.date
up_2018_2019.index = pd.to_datetime(up_2018_2019.index)
up_2018_2019 = up_2018_2019.resample('D').first()

down_2018_2019, m = waterdata.get_continuous(monitoring_location_id= f"USGS-{downstream_id}",
                                        parameter_code='00065',
                                        time=f"{years[2]}-01-01/{years[3]}-12-31",
                                        properties=["time","value","unit_of_measure"])
down_2018_2019.set_index("time",inplace=True)
down_2018_2019.index = down_2018_2019.index.date
down_2018_2019.index = pd.to_datetime(down_2018_2019.index)
down_2018_2019 = down_2018_2019.resample('D').first()

In [15]:
#2020-2021
up_2020_2021, m = waterdata.get_continuous(monitoring_location_id= f"USGS-{upstream_id}",
                                        parameter_code='00065',
                                        time=f"{years[4]}-01-01/{years[5]}-12-31",
                                        properties=["time","value","unit_of_measure"])
up_2020_2021.set_index("time",inplace=True)
up_2020_2021.index = up_2020_2021.index.date
up_2020_2021.index = pd.to_datetime(up_2020_2021.index)
up_2020_2021 = up_2020_2021.resample('D').first()

down_2020_2021, m = waterdata.get_continuous(monitoring_location_id= f"USGS-{downstream_id}",
                                        parameter_code='00065',
                                        time=f"{years[4]}-01-01/{years[5]}-12-31",
                                        properties=["time","value","unit_of_measure"])
down_2020_2021.set_index("time",inplace=True)
down_2020_2021.index = down_2020_2021.index.date
down_2020_2021.index = pd.to_datetime(down_2020_2021.index)
down_2020_2021 = down_2020_2021.resample('D').first()

In [17]:
#2022-2023
up_2022_2023, m = waterdata.get_continuous(monitoring_location_id= f"USGS-{upstream_id}",
                                        parameter_code='00065',
                                        time=f"{years[6]}-01-01/{years[7]}-12-31",
                                        properties=["time","value","unit_of_measure"])
up_2022_2023.set_index("time",inplace=True)
up_2022_2023.index = up_2022_2023.index.date
up_2022_2023.index = pd.to_datetime(up_2022_2023.index)
up_2022_2023 = up_2022_2023.resample('D').first()

down_2022_2023, m = waterdata.get_continuous(monitoring_location_id= f"USGS-{downstream_id}",
                                        parameter_code='00065',
                                        time=f"{years[6]}-01-01/{years[7]}-12-31",
                                        properties=["time","value","unit_of_measure"])
down_2022_2023.set_index("time",inplace=True)
down_2022_2023.index = down_2022_2023.index.date
down_2022_2023.index = pd.to_datetime(down_2022_2023.index)
down_2022_2023 = down_2022_2023.resample('D').first()

In [19]:
#2024-2025
up_2024_2025, m = waterdata.get_continuous(monitoring_location_id= f"USGS-{upstream_id}",
                                        parameter_code='00065',
                                        time=f"{years[8]}-01-01/{years[9]}-12-31",
                                        properties=["time","value","unit_of_measure"])
up_2024_2025.set_index("time",inplace=True)
up_2024_2025.index = up_2024_2025.index.date
up_2024_2025.index = pd.to_datetime(up_2024_2025.index)
up_2024_2025 = up_2024_2025.resample('D').first()

down_2024_2025, m = waterdata.get_continuous(monitoring_location_id= f"USGS-{downstream_id}",
                                        parameter_code='00065',
                                        time=f"{years[8]}-01-01/{years[9]}-12-31",
                                        properties=["time","value","unit_of_measure"])
down_2024_2025.set_index("time",inplace=True)
down_2024_2025.index = down_2024_2025.index.date
down_2024_2025.index = pd.to_datetime(down_2024_2025.index)
down_2024_2025 = down_2024_2025.resample('D').first()

In [31]:
#combining daily average to surface
surface_upstream = pd.concat([up_2016_2017,up_2018_2019,up_2020_2021,up_2022_2023,up_2024_2025])
surface_upstream.index.name = "Date"
surface_upstream = surface_upstream.drop(columns = ["unit_of_measure"])
surface_upstream = surface_upstream.rename(columns={"value":"Upstream to Surface (ft)"})

surface_downstream = pd.concat([down_2016_2017,down_2018_2019,down_2020_2021,down_2022_2023,down_2024_2025])
surface_downstream.index.name = "Date"
surface_downstream = surface_downstream.drop(columns = ["unit_of_measure"])
surface_downstream = surface_downstream.rename(columns={"value":"Downstream to Surface (ft)"})

to_surface = pd.merge(surface_upstream,surface_downstream,how='outer',on = "Date")

In [56]:
surface_downstream.isna().sum()

Downstream to Surface (ft)    60
dtype: int64

In [45]:
#merge and add the to surfae and to gauge values

discharge_depth = pd.merge(to_surface,datum,how='outer',on="Date")
discharge_depth["Upstream Depth (ft)"] = discharge_depth["Upstream Datum (ft)"] + discharge_depth["Upstream to Surface (ft)"]
discharge_depth["Downstream Depth (ft)"] = discharge_depth["Downstream Datum (ft)"] + discharge_depth["Downstream to Surface (ft)"]
discharge_depth = discharge_depth.drop(columns=["Upstream to Surface (ft)","Downstream to Surface (ft)","Upstream Datum (ft)","Downstream Datum (ft)"])

In [ ]:
cleaned = discharge_depth.dropna()
range = pd.date_range(start=cleaned.index.min(), end=cleaned.index.max(), freq='D')

missing_dates = range.difference(cleaned.index)
print("Missing dates:", missing_dates)